Tokenization  Challenge

Imports

In [22]:
%%capture
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os # For adjusting Keras settings.
os.environ['KERAS_BACKEND'] = 'jax' # Set a parameter for Keras.

# Packages used.
import jax.numpy as jnp # For defining matrices.
import keras # For adjusting Keras settings.
import pandas as pd # For loading the dataset.
import json
from datasets import load_dataset
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers, decoders


from ai_foundations import machine_learning # For defining and training MLPs.
from ai_foundations import visualizations # For visualizing data and boundaries.
from ai_foundations import training # For logging the loss during training.

1. Load the Dataset

In [25]:
# Loading the dataset for the Multilingual Tokenization Challenge
dataset = load_dataset("Similoluwa/african-multilingual-tokenizer-challenge")
train_ds = dataset["train"]
val_ds = dataset["validation"]

# Combine text rows to create a training iterator
def batch_iterator(batch_size=1000):
    for i in range(0, len(train_ds), batch_size):
        yield train_ds[i : i + batch_size]["text"]



2. Build Byte-Level BPE Tokenizer

Byte-Level BPE guarantees 0% unknown tokens because all unseen inputs
fall back to raw byte representations (0-255).

In [27]:
tokenizer = Tokenizer(models.BPE(unk_token=None))

# Normalize to NFC (as required by the challenge guidelines)
tokenizer.normalizer = normalizers.Sequence([normalizers.NFC()])

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

# Configure Trainer with strict 10,000 vocabulary budget
trainer = trainers.BpeTrainer(
    vocab_size=10000,
    special_tokens=["<PAD>", "<BOS>", "<EOS>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)

# Train the tokenizer
print("Training tokenizer on 240,000 passages...")
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)



Training tokenizer on 240,000 passages...


3. Evaluate Token Fertility & Losslessness

In [28]:

total_words = 0
total_tokens = 0
lossless_passes = 0

for row in val_ds:
    text = row["text"]
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids)

    # Check exact reconstruction
    if decoded == text:
        lossless_passes += 1

    words = len(text.split())
    tokens = len(encoded.ids)
    total_words += words
    total_tokens += tokens

fertility = total_tokens / total_words if total_words > 0 else 0
reconstruction_rate = (lossless_passes / len(val_ds)) * 100

print("\n--- Validation Results ---")
print(f"Token Fertility: {fertility:.4f} tokens/word")
print(f"Reconstruction Rate: {reconstruction_rate:.2f}% (Lossless check)")
print(f"Unknown Token Rate: 0.00% (Guaranteed by Byte-Level BPE)")


--- Validation Results ---
Token Fertility: 2.0313 tokens/word
Reconstruction Rate: 100.00% (Lossless check)
Unknown Token Rate: 0.00% (Guaranteed by Byte-Level BPE)


In [31]:

# Load and print the JSON formatted neatly
with open("tokenizer.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Print high-level keys
print("Keys in tokenizer.json:", list(data.keys()))
vocab = data["model"]["vocab"]
print(f"Total vocabulary size: {len(vocab)}")

# Print a small sample of the vocabulary
print("Vocabulary sample:", list(vocab.items())[:10])

Keys in tokenizer.json: ['version', 'truncation', 'padding', 'added_tokens', 'normalizer', 'pre_tokenizer', 'post_processor', 'decoder', 'model']
Total vocabulary size: 10000
Vocabulary sample: [('<PAD>', 0), ('<BOS>', 1), ('<EOS>', 2), ('!', 3), ('"', 4), ('#', 5), ('$', 6), ('%', 7), ('&', 8), ("'", 9)]


Evaluation Language Performance

In [33]:
import pandas as pd

results = []

for row in val_ds:
    text = row["text"]
    # Handle language field if available in the dataset schema
    lang = row.get("language", row.get("lang", "unknown"))

    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids)

    words = len(text.split())
    tokens = len(encoded.ids)

    results.append({
        "language": lang,
        "words": words,
        "tokens": tokens,
        "fertility": tokens / words if words > 0 else 0,
        "lossless": (decoded == text)
    })

df_res = pd.DataFrame(results)

# Group by language to see individual fertility scores
per_lang_summary = df_res.groupby("language").agg(
    avg_fertility=("fertility", "mean"),
    lossless_pct=("lossless", lambda x: x.mean() * 100)
).reset_index()

print(per_lang_summary)

  language  avg_fertility  lossless_pct
0       am       3.015782         100.0
1       en       1.935952         100.0
2       fr       2.049005         100.0
3       ha       1.798467         100.0
4       sw       2.013876         100.0
5       yo       2.312864         100.0
